In [1]:
from pathlib import Path
import json
import joblib
import pandas as pd
import numpy as np
import sys
import subprocess

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

print("Libraries imported successfully")

Libraries imported successfully


In [3]:
from pathlib import Path

CURRENT_DIR = Path.cwd()

# Search the current folder and its parent
possible_paths = [
    CURRENT_DIR / "models" / "day3_selected_model.joblib",
    CURRENT_DIR.parent / "models" / "day3_selected_model.joblib",
]

DAY3_MODEL_PATH = next(
    (path for path in possible_paths if path.exists()),
    None
)

if DAY3_MODEL_PATH is None:
    print("Day 3 model was not found.")
    print("\nChecked these locations:")
    for path in possible_paths:
        print(path)
else:
    PROJECT_DIR = DAY3_MODEL_PATH.parent.parent
    REPORTS_DIR = PROJECT_DIR / "reports"
    MODELS_DIR = PROJECT_DIR / "models"
    TESTS_DIR = CURRENT_DIR / "tests"

    REPORTS_DIR.mkdir(parents=True, exist_ok=True)
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    TESTS_DIR.mkdir(parents=True, exist_ok=True)

    print("Project folder:", PROJECT_DIR)
    print("Day 3 model found:", DAY3_MODEL_PATH)

Day 3 model was not found.

Checked these locations:
c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day4-Error inspection and tests\models\day3_selected_model.joblib
c:\Users\Work\Desktop\Testable ML training and inference pipeline\models\day3_selected_model.joblib


In [4]:
from pathlib import Path

search_root = Path.home() / "Desktop"

for path in search_root.glob("**/day3_selected_model.joblib"):
    print("Found model:", path)

Found model: C:\Users\Work\Desktop\Testable ML training and inference pipeline\Day3-Baseline training and evaluation\models\day3_selected_model.joblib


In [5]:
DAY3_MODEL_PATH = Path(r"PASTE_THE_EXACT_PATH_HERE")

In [12]:
from pathlib import Path
import joblib

# Search the Desktop for the saved Day 3 model
search_root = Path.home() / "Desktop"

matches = list(
    search_root.glob("**/day3_selected_model.joblib")
)

if matches:
    print("Found model file(s):")
    for i, path in enumerate(matches, start=1):
        print(f"{i}. {path}")

    DAY3_MODEL_PATH = matches[0]
    selected_model = joblib.load(DAY3_MODEL_PATH)

    print("\nModel loaded successfully!")
    print("Using:", DAY3_MODEL_PATH)

else:
    print("Day 3 model was not found on the Desktop.")
    print("We need to check where the Day 3 model was saved.")

Found model file(s):
1. C:\Users\Work\Desktop\Testable ML training and inference pipeline\Day3-Baseline training and evaluation\models\day3_selected_model.joblib

Model loaded successfully!
Using: C:\Users\Work\Desktop\Testable ML training and inference pipeline\Day3-Baseline training and evaluation\models\day3_selected_model.joblib


In [8]:
from pathlib import Path

search_root = Path.home() / "Desktop"

for path in search_root.glob("**/models/*.joblib"):
    print(path)

C:\Users\Work\Desktop\AIML-Training\Reproducible tabular prediction workflow\Day5-End to end ML capstone\models\breast_cancer_pipeline.joblib
C:\Users\Work\Desktop\Reproducible tabular prediction workflow\Day5-End to end ML capstone\models\breast_cancer_pipeline.joblib
C:\Users\Work\Desktop\Testable ML training and inference pipeline\Day2-Preprocessing and leakage-safe pipeline\models\day2_ml_pipeline.joblib
C:\Users\Work\Desktop\Testable ML training and inference pipeline\Day3-Baseline training and evaluation\models\day3_selected_model.joblib


In [13]:
from pathlib import Path
import joblib

DAY3_MODEL_PATH = Path(
    r"C:\Users\Work\Desktop\Testable ML training and inference pipeline\Day3-Baseline training and evaluation\models\day3_selected_model.joblib"
)

assert DAY3_MODEL_PATH.exists(), "Day 3 model file not found!"

selected_model = joblib.load(DAY3_MODEL_PATH)

print("Day 3 model loaded successfully!")
print("Model path:", DAY3_MODEL_PATH)

Day 3 model loaded successfully!
Model path: C:\Users\Work\Desktop\Testable ML training and inference pipeline\Day3-Baseline training and evaluation\models\day3_selected_model.joblib


In [14]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

dataset = load_breast_cancer(as_frame=True)

X = dataset.data
y = dataset.target
FEATURE_COLUMNS = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

y_pred = selected_model.predict(X_test)

print("Dataset shape:", X.shape)
print("Training records:", len(X_train))
print("Testing records:", len(X_test))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Malignant", "Benign"],
    zero_division=0
))

Dataset shape: (569, 30)
Training records: 455
Testing records: 114

Confusion matrix:
[[41  1]
 [ 1 71]]

Classification report:
              precision    recall  f1-score   support

   Malignant       0.98      0.98      0.98        42
      Benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



In [15]:
import pandas as pd

# Create a table of actual and predicted results
error_df = X_test.copy()

error_df["actual"] = y_test
error_df["predicted"] = y_pred

error_df["actual_label"] = error_df["actual"].map({
    0: "Malignant",
    1: "Benign"
})

error_df["predicted_label"] = error_df["predicted"].map({
    0: "Malignant",
    1: "Benign"
})

error_df["is_error"] = error_df["actual"] != error_df["predicted"]

def classify_error(row):
    if not row["is_error"]:
        return "Correct"
    elif row["actual"] == 1 and row["predicted"] == 0:
        return "False Positive"
    else:
        return "False Negative"

error_df["error_type"] = error_df.apply(classify_error, axis=1)

# Display only incorrect predictions
errors_only = error_df[error_df["is_error"]]

print("Total test records:", len(error_df))
print("Incorrect predictions:", len(errors_only))
print("\nError types:")
print(error_df["error_type"].value_counts())

print("\nIncorrect prediction details:")
display(errors_only[[
    "actual_label",
    "predicted_label",
    "error_type"
]])

Total test records: 114
Incorrect predictions: 2

Error types:
error_type
Correct           112
False Positive      1
False Negative      1
Name: count, dtype: int64

Incorrect prediction details:


,actual_label,predicted_label,error_type
541,Benign,Malignant,False Positive
73,Malignant,Benign,False Negative


In [16]:
from pathlib import Path

# Save reports in your Day 4 folder
PROJECT_DIR = Path.cwd()
REPORTS_DIR = PROJECT_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

error_path = REPORTS_DIR / "day4_error_analysis.csv"
summary_path = REPORTS_DIR / "day4_error_summary.csv"

errors_only.to_csv(error_path, index_label="record_id")

summary = (
    error_df["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .reset_index(name="count")
)

summary.to_csv(summary_path, index=False)

print("Error report saved:", error_path)
print("Error summary saved:", summary_path)
print("\nSummary:")
display(summary)

Error report saved: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day4-Error inspection and tests\reports\day4_error_analysis.csv
Error summary saved: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day4-Error inspection and tests\reports\day4_error_summary.csv

Summary:


,error_type,count
0,Correct,112
1,False Positive,1
2,False Negative,1


In [17]:
from pathlib import Path

PROJECT_DIR = Path.cwd()

validator_code = '''
import pandas as pd

def validate_input(data, feature_columns):
    if not isinstance(data, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if data.empty:
        raise ValueError("Input must contain at least one row.")

    if data.columns.duplicated().any():
        raise ValueError("Duplicate feature columns are not allowed.")

    missing = set(feature_columns) - set(data.columns)
    extra = set(data.columns) - set(feature_columns)

    if missing:
        raise ValueError(f"Missing required features: {sorted(missing)}")

    if extra:
        raise ValueError(f"Unexpected features: {sorted(extra)}")

    if not all(
        pd.api.types.is_numeric_dtype(data[col])
        for col in feature_columns
    ):
        raise TypeError("All features must be numeric.")

    return data[feature_columns]
'''

validator_path = PROJECT_DIR / "day4_inference.py"
validator_path.write_text(validator_code, encoding="utf-8")

print("Created:", validator_path)

Created: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day4-Error inspection and tests\day4_inference.py


In [18]:
TESTS_DIR = PROJECT_DIR / "tests"
TESTS_DIR.mkdir(parents=True, exist_ok=True)

test_code = '''
import pandas as pd
import pytest
from sklearn.datasets import load_breast_cancer

from day4_inference import validate_input


@pytest.fixture
def sample_data():
    dataset = load_breast_cancer(as_frame=True)
    return dataset.data.iloc[[0]].copy(), dataset.data.columns.tolist()


def test_valid_input(sample_data):
    row, features = sample_data
    result = validate_input(row, features)
    assert result.shape == (1, len(features))


def test_reordered_columns(sample_data):
    row, features = sample_data
    result = validate_input(row[features[::-1]], features)
    assert result.columns.tolist() == features


def test_missing_feature_rejected(sample_data):
    row, features = sample_data
    row = row.drop(columns=[features[0]])
    with pytest.raises(ValueError, match="Missing required features"):
        validate_input(row, features)


def test_extra_feature_rejected(sample_data):
    row, features = sample_data
    row["unexpected_feature"] = 1
    with pytest.raises(ValueError, match="Unexpected features"):
        validate_input(row, features)


def test_empty_input_rejected(sample_data):
    row, features = sample_data
    with pytest.raises(ValueError, match="at least one row"):
        validate_input(row.iloc[0:0], features)


def test_duplicate_columns_rejected(sample_data):
    row, features = sample_data
    row.columns = features[:-1] + [features[0]]
    with pytest.raises(ValueError, match="Duplicate feature columns"):
        validate_input(row, features)


def test_nonnumeric_feature_rejected(sample_data):
    row, features = sample_data
    row[features[0]] = "not-a-number"
    with pytest.raises(TypeError, match="must be numeric"):
        validate_input(row, features)


def test_non_dataframe_rejected(sample_data):
    _, features = sample_data
    with pytest.raises(TypeError, match="pandas DataFrame"):
        validate_input([[1, 2, 3]], features)
'''

test_file = TESTS_DIR / "test_day4_inference.py"
test_file.write_text(test_code, encoding="utf-8")

print("Created test file:", test_file)

Created test file: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day4-Error inspection and tests\tests\test_day4_inference.py


In [19]:
%pip install pytest

Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/3 [pluggy]
   ------------- -------------------------- 1/3 [iniconfig]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   -------------------------- ------------- 2/3 [pytest]
   ---


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-v",
        str(test_file)
    ],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Some tests failed. Check the test output.")

print("All automated tests passed!")

============================= test session starts =============================
platform win32 -- Python 3.10.9, pytest-9.1.1, pluggy-1.7.0 -- c:\Program Files\Python310\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day4-Error inspection and tests
plugins: anyio-4.15.1, platformdirs-4.12.3
collecting ... collected 8 items

tests/test_day4_inference.py::test_valid_input PASSED                    [ 12%]
tests/test_day4_inference.py::test_reordered_columns PASSED              [ 25%]
tests/test_day4_inference.py::test_missing_feature_rejected PASSED       [ 37%]
tests/test_day4_inference.py::test_extra_feature_rejected PASSED         [ 50%]
tests/test_day4_inference.py::test_empty_input_rejected PASSED           [ 62%]
tests/test_day4_inference.py::test_duplicate_columns_rejected PASSED     [ 75%]
tests/test_day4_inference.py::test_nonnumeric_feature_rejected PASSED    [ 87%]
tests/test_day4_inference.py::test_non_dataframe_re

In [21]:
import json
import platform
from pathlib import Path

PROJECT_DIR = Path.cwd()
MODELS_DIR = PROJECT_DIR / "models"
REPORTS_DIR = PROJECT_DIR / "reports"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

metadata = {
    "project": "Testable ML Training and Inference Pipeline",
    "day": 4,
    "model_type": type(selected_model).__name__,
    "dataset": "Breast Cancer Wisconsin dataset",
    "dataset_rows": int(X.shape[0]),
    "feature_count": len(FEATURE_COLUMNS),
    "feature_columns": FEATURE_COLUMNS,
    "target_column": "target",
    "target_labels": {
        "0": "malignant",
        "1": "benign"
    },
    "test_size": 0.20,
    "random_state": 42,
    "stratified_split": True,
    "python_version": platform.python_version(),
    "validation_tests_passed": 8
}

metadata_path = MODELS_DIR / "day4_model_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=4)

print("Metadata saved:", metadata_path)
print("Feature count:", metadata["feature_count"])

Metadata saved: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day4-Error inspection and tests\models\day4_model_metadata.json
Feature count: 30


In [25]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Generate predictions
y_pred = selected_model.predict(X_test)

# Calculate accuracy and confusion matrix
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

# Create a table of incorrect predictions
error_df = X_test.copy()
error_df["actual"] = y_test
error_df["predicted"] = y_pred

errors = error_df[error_df["actual"] != error_df["predicted"]].copy()

# Class 0 = malignant, class 1 = benign
false_positives = int(((y_test == 1) & (y_pred == 0)).sum())
false_negatives = int(((y_test == 0) & (y_pred == 1)).sum())

print("Accuracy:", round(accuracy * 100, 2), "%")
print("Incorrect predictions:", len(errors))
print("False positives:", false_positives)
print("False negatives:", false_negatives)
print("Confusion matrix:\n", cm)

Accuracy: 98.25 %
Incorrect predictions: 2
False positives: 1
False negatives: 1
Confusion matrix:
 [[41  1]
 [ 1 71]]


In [26]:
from pathlib import Path
from sklearn.metrics import classification_report

report_lines = [
    "# Day 4 — Error Inspection and Tests",
    "",
    "## 1. Project overview",
    "- Dataset: Breast Cancer Wisconsin dataset",
    f"- Total records: {len(X)}",
    f"- Features: {len(FEATURE_COLUMNS)}",
    f"- Model: {type(selected_model).__name__}",
    f"- Training records: {len(X_train)}",
    f"- Test records: {len(X_test)}",
    "- Split: 80% training / 20% testing",
    "- Random state: 42",
    "- Stratified split: Yes",
    "",
    "## 2. Evaluation results",
    f"- Test accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)",
    f"- Incorrect predictions: {len(errors)}",
    f"- False positives for malignant class: {false_positives}",
    f"- False negatives for malignant class: {false_negatives}",
    "",
    "A false positive means a benign case predicted as malignant.",
    "A false negative means a malignant case predicted as benign.",
    "",
    "### Confusion matrix",
    "Rows are actual labels; columns are predicted labels.",
    "Label order: 0 = malignant, 1 = benign.",
    "",
    "    " + str(cm).replace("\n", "\n    "),
    "",
    "### Classification report",
    "```text",
    classification_report(
        y_test,
        y_pred,
        target_names=["malignant", "benign"],
        zero_division=0
    ).rstrip(),
    "```",
    "",
    "## 3. Input validation",
    "All 8 automated tests passed:",
    "- Valid input accepted",
    "- Reordered columns handled",
    "- Missing feature rejected",
    "- Extra feature rejected",
    "- Empty input rejected",
    "- Duplicate columns rejected",
    "- Non-numeric feature rejected",
    "- Non-DataFrame input rejected",
    "",
    "## 4. Reproducibility",
    "- Dataset: scikit-learn Breast Cancer Wisconsin dataset",
    "- Test size: 0.20",
    "- Random state: 42",
    "- Stratification: enabled",
    f"- Python version: {metadata['python_version']}",
    "",
    "## 5. Limitations",
    "- This is a public demonstration dataset, not company data.",
    "- Test-set results alone do not establish real-world clinical performance.",
    "- Numeric type validation does not ensure values are realistic.",
    "- This model must not be used for medical decisions.",
    "",
    "## 6. Output files",
    "- models/day4_model_metadata.json",
    "- models/day4_model_bundle.joblib",
    "- reports/day4_error_analysis.csv",
    "- reports/day4_error_summary.csv",
    "- DAY4_ERROR_ANALYSIS_REPORT.md",
]

report = "\n".join(str(line) for line in report_lines) + "\n"

report_path = PROJECT_DIR / "DAY4_ERROR_ANALYSIS_REPORT.md"
report_path.write_text(report, encoding="utf-8")

print("Final report saved:", report_path)
print("Day 4 report created successfully!")

Final report saved: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day4-Error inspection and tests\DAY4_ERROR_ANALYSIS_REPORT.md
Day 4 report created successfully!


In [30]:
import joblib

MODELS_DIR.mkdir(parents=True, exist_ok=True)

bundle = {
    "model": selected_model,
    "feature_columns": FEATURE_COLUMNS,
    "metadata": metadata
}

bundle_path = MODELS_DIR / "day4_model_bundle.joblib"
joblib.dump(bundle, bundle_path)

print("Model bundle saved:", bundle_path)
print("File exists:", bundle_path.exists())

Model bundle saved: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day4-Error inspection and tests\models\day4_model_bundle.joblib
File exists: True


In [31]:
expected_files = [
    MODELS_DIR / "day4_model_metadata.json",
    MODELS_DIR / "day4_model_bundle.joblib",
    REPORTS_DIR / "day4_error_analysis.csv",
    REPORTS_DIR / "day4_error_summary.csv",
    PROJECT_DIR / "DAY4_ERROR_ANALYSIS_REPORT.md",
]

for file_path in expected_files:
    status = "PASS" if file_path.exists() else "MISSING"
    print(f"{status} - {file_path.name}")

PASS - day4_model_metadata.json
PASS - day4_model_bundle.joblib
PASS - day4_error_analysis.csv
PASS - day4_error_summary.csv
PASS - DAY4_ERROR_ANALYSIS_REPORT.md
